In [0]:
DROP TABLE IF EXISTS workspace.tmw_trabalhofinal.abt_ativacao;
CREATE TABLE IF NOT EXISTS workspace.tmw_trabalhofinal.abt_ativacao 

WITH tb_ativacao AS (
    SELECT  DISTINCT
            IdCliente,
            DATE(DtCriacao) AS dtAtivacao
    FROM workspace.tmw_loyalty.transacoes
),

tb_resposta AS (
    SELECT  t1.IdCliente,
            t1.dtRef,
            COUNT(t2.dtativacao) AS qtdAtivacoes,
            MAX(CASE WHEN t2.dtativacao IS NOT NULL THEN 1 ELSE 0 END) AS flAtivacao
    FROM    workspace.tmw_trabalhofinal.fs_pontos AS t1
            LEFT JOIN tb_ativacao AS t2
                ON t1.IdCliente = t2.idcliente
                AND t1.dtRef <= t2.dtAtivacao
                AND t2.dtAtivacao <= (t1.dtRef + INTERVAL 28 DAYS)
    GROUP BY ALL
    ORDER BY t1.IdCliente, 
            t1.dtRef
),

tb_features AS ( 
    SELECT  t2.*,
            t3.qtdCursosIniciados,
            t3.qtdCursosFinalizados,
            t3.github_2025,
            t3.python_2025,
            t3.sql_2025,
            t3.pandas_2025,
            t3.estatistica_2025,
            t3.avgTempoInicioUltimo,
            t3.avgTempoInicioFim,
            t3.qtEpisodiosConcluidos28d,
            t1.flAtivacao
    FROM    tb_resposta AS t1
            LEFT JOIN workspace.tmw_trabalhofinal.fs_pontos AS t2
                ON t1.idcliente = t2.IdCliente
                AND t1.dtref=t2.dtRef
            LEFT JOIN workspace.tmw_trabalhofinal.fs_cursos AS t3
                ON t1.idcliente = t3.IdCliente
                AND t1.dtref=t3.dtRef
),

tb_sample AS (
    SELECT  *
    FROM    tb_features
    WHERE   dtRef <= '2026-05-01'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY idCliente ORDER BY RAND()) = 1

    UNION ALL

    SELECT  *
    FROM    tb_features
    WHERE   dtRef = '2026-06-01'
)

SELECT  *
FROM    tb_sample;